In [5]:
"""
Flow Logs - Bronze to Silver (BATCHED)
Processes data in date-based batches to avoid memory issues

UPDATED: Now writes to Delta table instead of Parquet volume
"""

from pyspark.sql import functions as F
from datetime import datetime, timedelta

# ============================================================================
# CONFIGURATION
# ============================================================================

BRONZE_PATH = "/Volumes/security_lake/default/bronze_oci_flow_logs/data"

# NEW: Delta table instead of Parquet path
CATALOG = "gitrepo"
SCHEMA = "default"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_flow_logs"

# State tracking (unchanged)
STATE_PATH = "/Volumes/gitrepo/default/git_oci_aidp_silver/flow_logs/state"

# Internal IP Prefixes
INTERNAL_IP_PREFIXES = ["10.", "192.168.", "169.254."]

# BATCH SIZE - Process N days at a time
BATCH_DAYS = 7  # Process 7 days per run

# Spark tuning - more conservative for large data
spark.conf.set("spark.sql.shuffle.partitions", "400")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

print("=" * 70)
print("FLOW LOGS - BRONZE TO SILVER (DELTA TABLE)")
print("=" * 70)
print(f"Bronze Path: {BRONZE_PATH}")
print(f"Silver Table: {SILVER_TABLE}")
print(f"Batch Size: {BATCH_DAYS} days")
print(f"Internal IP Prefixes: {INTERNAL_IP_PREFIXES}")
print("=" * 70)

# ============================================================================
# LOAD STATE
# ============================================================================

print("\n[STEP 1] Loading state...")

last_date = None
try:
    state_df = spark.read.parquet(STATE_PATH)
    row = state_df.select("last_ingest_date").orderBy(F.col("last_ingest_date").desc()).first()
    last_date = row[0] if row else None
except:
    last_date = None

print(f"Last processed date: {last_date}")

# ============================================================================
# READ BRONZE
# ============================================================================

print("\n[STEP 2] Reading bronze layer...")

bronze_df = (spark.read.format("delta").load(BRONZE_PATH)
             .withColumn("ingest_date", F.to_date("ingest_time")))

# Get date range
if last_date is None:
    # First run - get earliest date
    min_date = bronze_df.select(F.min("ingest_date")).first()[0]
    print(f"First run - starting from: {min_date}")
    start_date = min_date
else:
    # Continue from last processed
    start_date = last_date + timedelta(days=1)
    print(f"Continuing from: {start_date}")

# Calculate end date for this batch
end_date = start_date + timedelta(days=BATCH_DAYS - 1)

# Filter to batch date range
bronze_batch = bronze_df.filter(
    (F.col("ingest_date") >= F.lit(start_date)) &
    (F.col("ingest_date") <= F.lit(end_date))
)

batch_count = bronze_batch.count()

if batch_count == 0:
    print(f"\n✓ No data for {start_date} to {end_date}")
    print("✓ Silver is up to date")
    
    try:
        silver_df = spark.table(SILVER_TABLE)
        print(f"Current silver: {silver_df.count():,} records")
    except:
        print("Silver table is empty")

else:
    print(f"\nProcessing {batch_count:,} records from {start_date} to {end_date}")
    
    # ========================================================================
    # PARSE JSON
    # ========================================================================
    
    print("\n[STEP 3] Parsing JSON...")
    
    silver_df = (
        bronze_batch
        # Core flow fields
        .withColumn("src_ip", F.get_json_object("raw_json", "$.data.sourceAddress"))
        .withColumn("dst_ip", F.get_json_object("raw_json", "$.data.destinationAddress"))
        .withColumn("src_port", F.get_json_object("raw_json", "$.data.sourcePort").cast("int"))
        .withColumn("dst_port", F.get_json_object("raw_json", "$.data.destinationPort").cast("int"))
        .withColumn("protocol_num", F.get_json_object("raw_json", "$.data.protocol").cast("int"))
        .withColumn("protocol_name", F.get_json_object("raw_json", "$.data.protocolName"))
        .withColumn("action", F.get_json_object("raw_json", "$.data.action"))
        .withColumn("status", F.get_json_object("raw_json", "$.data.status"))
        .withColumn("bytes", F.get_json_object("raw_json", "$.data.bytesOut").cast("bigint"))
        .withColumn("packets", F.get_json_object("raw_json", "$.data.packets").cast("bigint"))
        
        # Timing
        .withColumn("start_time_epoch", F.get_json_object("raw_json", "$.data.startTime").cast("bigint"))
        .withColumn("end_time_epoch", F.get_json_object("raw_json", "$.data.endTime").cast("bigint"))
        .withColumn("start_time", F.from_unixtime(F.col("start_time_epoch")))
        .withColumn("end_time", F.from_unixtime(F.col("end_time_epoch")))
        .withColumn("duration_seconds", F.col("end_time_epoch") - F.col("start_time_epoch"))
        
        # OCI metadata
        .withColumn("compartment_id", F.get_json_object("raw_json", "$.oracle.compartmentid"))
        .withColumn("tenancy_id", F.get_json_object("raw_json", "$.oracle.tenantid"))
        .withColumn("vcn_id", F.get_json_object("raw_json", "$.oracle.vcnOcid"))
        .withColumn("subnet_id", F.get_json_object("raw_json", "$.oracle.vnicsubnetocid"))
        .withColumn("vnic_id", F.get_json_object("raw_json", "$.oracle.vnicocid"))
        .withColumn("instance_id", F.get_json_object("raw_json", "$.oracle.instanceOcid"))
        .withColumn("public_ipv4", F.get_json_object("raw_json", "$.oracle.publicIpv4"))
        
        # Top-level
        .withColumn("event_time_str", F.get_json_object("raw_json", "$.time"))
        .withColumn("event_time", F.to_timestamp("event_time_str"))
        .withColumn("log_type", F.get_json_object("raw_json", "$.type"))
        .withColumn("flow_id", F.get_json_object("raw_json", "$.data.flowid"))
    )
    
    # Filter valid
    silver_df = silver_df.filter(F.col("src_ip").isNotNull() & F.col("dst_ip").isNotNull())
    
    # Derived fields
    if INTERNAL_IP_PREFIXES:
        conditions = [F.col("src_ip").startswith(prefix) for prefix in INTERNAL_IP_PREFIXES]
        is_internal_condition = conditions[0]
        for condition in conditions[1:]:
            is_internal_condition = is_internal_condition | condition
    else:
        is_internal_condition = F.lit(False)
    
    silver_df = (silver_df
        .withColumn("is_rejected", F.when(F.col("action") == "REJECT", True).otherwise(False))
        .withColumn("is_internal", F.when(is_internal_condition, True).otherwise(False))
    )
    
    parsed_count = silver_df.count()
    print(f"✓ Parsed {parsed_count:,} valid records")
    
    # ========================================================================
    # WRITE TO DELTA TABLE (instead of Parquet)
    # ========================================================================
    
    print("\n[STEP 4] Writing to Delta table...")
    
    (
        silver_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(SILVER_TABLE)
    )
    
    print(f"✓ Data appended to {SILVER_TABLE}")
    
    # ========================================================================
    # UPDATE STATE
    # ========================================================================
    
    print("\n[STEP 5] Updating state...")
    
    (spark.createDataFrame([(end_date,)], ["last_ingest_date"])
     .write.mode("overwrite")
     .parquet(STATE_PATH))
    
    print(f"✓ State updated to: {end_date}")
    
    # ========================================================================
    # STATS
    # ========================================================================
    
    print("\n" + "=" * 70)
    print("✓ BATCH COMPLETED")
    print("=" * 70)
    print(f"  Date range: {start_date} to {end_date}")
    print(f"  Records processed: {parsed_count:,}")
    
    # Check if more data exists
    next_start = end_date + timedelta(days=1)
    remaining = bronze_df.filter(F.col("ingest_date") >= F.lit(next_start)).count()
    
    if remaining > 0:
        print(f"  Remaining bronze records: {remaining:,}")
        print(f"\n📋 Run again to process next {BATCH_DAYS} days")
    else:
        silver_total = spark.table(SILVER_TABLE).count()
        print(f"\n✓ ALL DATA PROCESSED!")
        print(f"  Total silver records: {silver_total:,}")
    
    print("=" * 70)

print("\n✓ Done")

FLOW LOGS - BRONZE TO SILVER (BATCHED)
Batch Size: 7 days
Internal IP Prefixes: ['10.', '192.168.', '169.254.']

[STEP 1] Loading state...


Last processed date: 2026-01-11

[STEP 2] Reading bronze layer...


Continuing from: 2026-01-12



Processing 38,632,046 records from 2026-01-12 to 2026-01-18

[STEP 3] Parsing JSON...


✓ Parsed 38,395,935 valid records

[STEP 4] Writing to silver...


✓ Written to silver

[STEP 5] Updating state...


✓ State updated to: 2026-01-18

✓ BATCH COMPLETED
  Date range: 2026-01-12 to 2026-01-18
  Records processed: 38,395,935


  Remaining bronze records: 44,608,534

📋 Run again to process next 7 days

✓ Done


In [ ]:
from pyspark.sql import functions as F

SILVER_PATH = "/Volumes/gitrepo/default/git_oci_aidp_silver/flow_logs/data"

silver_df = spark.read.parquet(SILVER_PATH)

print(f"Total Records: {silver_df.count():,}")

# Sample 5 records with key fields
silver_df.select(
    "event_time",
    "src_ip", 
    "dst_ip",
    "src_port",
    "dst_port",
    "protocol_name",
    "action",
    "bytes",
    "packets",
    "is_rejected",
    "is_internal"
).show(5, truncate=False)

# Quick stats
silver_df.groupBy("action").count().show()
silver_df.groupBy("protocol_name").count().orderBy(F.desc("count")).show(10)

Total Records: 161,667,060


+-------------------+---------------+--------------+--------+--------+-------------+------+-----+-------+-----------+-----------+
|event_time         |src_ip         |dst_ip        |src_port|dst_port|protocol_name|action|bytes|packets|is_rejected|is_internal|
+-------------------+---------------+--------------+--------+--------+-------------+------+-----+-------+-----------+-----------+
|2026-01-05 02:39:06|43.208.26.130  |192.168.1.47  |8       |0       |ICMP         |ACCEPT|68   |1      |false      |false      |
|2025-12-24 01:45:25|10.0.0.42      |140.245.48.111|22      |40664   |TCP          |ACCEPT|40   |1      |false      |true       |
|2026-01-05 02:38:07|147.185.133.137|192.168.1.47  |54909   |9676    |TCP          |REJECT|44   |1      |true       |false      |
|2025-12-24 01:45:18|98.81.92.108   |10.0.0.42     |0       |8       |ICMP         |ACCEPT|136  |2      |false      |false      |
|2026-01-05 02:39:06|43.208.23.202  |192.168.1.47  |8       |0       |ICMP         |ACCEPT